In [ ]:
# =====================================================================
# CELL 1: Install & Import
# =====================================================================
!pip install stable-baselines3 sb3-contrib -q

import os, csv, time, shutil
import numpy as np
import torch
import torch.nn as nn
from scipy.ndimage import binary_dilation
from collections import deque

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.monitor import Monitor
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticCnnPolicy

print(f'CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# =====================================================================
# CELL 2: Đường dẫn — CHỈ SỬA Ở ĐÂY
# =====================================================================
# Input (file cũ từ Kaggle dataset)
INPUT_CHECKPOINT  = '/kaggle/input/notebooks/hauuto/quintetx-rl-resnet6-2/ppo_caro_resnet/caro_resnet_900000_steps.zip'
INPUT_STATE       = '/kaggle/input/notebooks/hauuto/quintetx-rl-resnet6-2/ppo_caro_resnet/curriculum_state.npy'
INPUT_CSV         = '/kaggle/input/notebooks/hauuto/quintetx-rl-resnet6-2/ppo_caro_resnet/training_log.csv'

# Output (working dir)
SAVE_DIR      = '/kaggle/working/ppo_caro_resnet/'
MODEL_PATH    = os.path.join(SAVE_DIR, 'caro_resnet.zip')
STATE_PATH    = os.path.join(SAVE_DIR, 'curriculum_state.npy')
CSV_PATH      = os.path.join(SAVE_DIR, 'training_log.csv')
PLOT_PATH     = os.path.join(SAVE_DIR, 'training_report.png')
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy file cũ vào working (chỉ nếu chưa có)
for src, dst in [(INPUT_CHECKPOINT, MODEL_PATH),
                 (INPUT_STATE,      STATE_PATH),
                 (INPUT_CSV,        CSV_PATH)]:
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'📥 Copied: {os.path.basename(src)}')
    elif os.path.exists(dst):
        print(f'✅ Already exists: {os.path.basename(dst)}')
    else:
        print(f'❌ MISSING: {src}')

In [ ]:
# =====================================================================
# CELL 3: Game constants & Environment
# =====================================================================
BOARD_SIZE  = 40
WIN_LEN     = 5
MAX_MOVES   = 600
DIRECTIONS  = [(1,0),(0,1),(1,1),(1,-1)]

REWARD_WIN   =  1.0
REWARD_LOSE  = -1.0
REWARD_BLOCK =  0.05
REWARD_SCALE =  1e-4
STEP_PENALTY = -0.001

_RADIUS          = 3
_DILATION_STRUCT = np.ones((2*_RADIUS+1, 2*_RADIUS+1), dtype=bool)

def check_win(board, r, c, player):
    for dr, dc in DIRECTIONS:
        cnt = 1
        nr, nc = r+dr, c+dc
        while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==player: cnt+=1; nr+=dr; nc+=dc
        nr, nc = r-dr, c-dc
        while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==player: cnt+=1; nr-=dr; nc-=dc
        if cnt >= WIN_LEN: return True
    return False

def static_evaluate(board, player, active_cells):
    score = 0.0
    for r, c in active_cells:
        p = board[r, c]
        if p == 0: continue
        mine = (p == player)
        for dr, dc in DIRECTIONS:
            pr, pc = r-dr, c-dc
            if 0<=pr<BOARD_SIZE and 0<=pc<BOARD_SIZE and board[pr,pc]==p: continue
            cnt = 1; nr, nc = r+dr, c+dc
            while 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==p: cnt+=1; nr+=dr; nc+=dc
            ends = 0
            if 0<=pr<BOARD_SIZE and 0<=pc<BOARD_SIZE and board[pr,pc]==0: ends+=1
            if 0<=nr<BOARD_SIZE and 0<=nc<BOARD_SIZE and board[nr,nc]==0: ends+=1
            if cnt >= WIN_LEN:   base = 200000
            elif ends == 0:      continue
            else:
                base = {1:10, 2:100, 3:1000, 4:100000}.get(cnt, 0)
                if ends == 1: base //= 4
            score += base if mine else -float(base * 2.0)
    return score

class CaroEnv(gym.Env):
    def __init__(self, opponent_level=0):
        super().__init__()
        self.opponent_level = opponent_level
        self.action_space = spaces.Discrete(BOARD_SIZE * BOARD_SIZE)
        self.observation_space = spaces.Box(0, 1, shape=(3, BOARD_SIZE, BOARD_SIZE), dtype=np.uint8)
        self.agent_player, self.bot_player = 1, 2
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
        self._obs_buf = np.zeros((3, BOARD_SIZE, BOARD_SIZE), dtype=np.uint8)
        self._occupied = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=bool)
        self._dilated_flat = np.zeros(BOARD_SIZE * BOARD_SIZE, dtype=bool)

    def set_opponent_level(self, level):
        self.opponent_level = level; self.last_score = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.board.fill(0)
        self._occupied.fill(False)
        self.move_count, self.last_score, self.last_opp_move = 0, 0.0, None
        self.active_cells = set()
        self.agent_player = np.random.choice([1, 2])
        self.bot_player   = 2 if self.agent_player == 1 else 1
        if self.bot_player == 1:
            center = BOARD_SIZE // 2
            self.board[center, center] = self.bot_player
            self._occupied[center, center] = True
            self.active_cells.add((center, center))
            self.last_opp_move = center * BOARD_SIZE + center
            self.move_count += 1
            self.last_score = static_evaluate(self.board, self.agent_player, self.active_cells)
        return self._obs(), {}

    def _obs(self):
        self._obs_buf.fill(0)
        self._obs_buf[0] = (self.board == self.agent_player)
        self._obs_buf[1] = (self.board == self.bot_player)
        if self.last_opp_move is not None:
            r, c = divmod(self.last_opp_move, BOARD_SIZE)
            self._obs_buf[2, r, c] = 1
        return self._obs_buf.copy()

    def _is_winning_move(self, player, r, c):
        if self.board[r, c] != 0: return False
        self.board[r, c] = player
        result = check_win(self.board, r, c, player)
        self.board[r, c] = 0
        return result

    def _get_opponent_move(self):
        masks = self.action_masks()
        valid = np.where(masks)[0]
        if len(valid) == 0: return BOARD_SIZE * BOARD_SIZE // 2 + BOARD_SIZE // 2
        lv = self.opponent_level
        if lv >= 1:
            for a in valid:
                r, c = divmod(a, BOARD_SIZE)
                if self._is_winning_move(self.bot_player, r, c): return a
        if lv >= 2:
            if lv >= 3 or np.random.rand() < 0.5:
                for a in valid:
                    r, c = divmod(a, BOARD_SIZE)
                    if self._is_winning_move(self.agent_player, r, c): return a
        if np.random.rand() < max(0.0, 1.0 - lv * 0.15): return np.random.choice(valid)
        candidates = valid if len(valid) <= 40 else np.random.choice(valid, 40, replace=False)
        best_score, best_a = -float('inf'), candidates[0]
        for a in candidates:
            r, c = divmod(a, BOARD_SIZE); self.board[r, c] = self.bot_player
            score = static_evaluate(self.board, self.bot_player, self.active_cells | {(r, c)})
            self.board[r, c] = 0
            if score > best_score: best_score, best_a = score, a
        return best_a

    def step(self, action):
        x, y = divmod(action, BOARD_SIZE)
        if self.board[x, y] != 0: return self._obs(), REWARD_LOSE, True, False, {'is_success': False}
        is_blocking = self._is_winning_move(self.bot_player, x, y)
        self.board[x, y] = self.agent_player
        self._occupied[x, y] = True
        self.active_cells.add((x, y))
        self.move_count += 1
        if check_win(self.board, x, y, self.agent_player): return self._obs(), REWARD_WIN, True, False, {'is_success': True, 'reason': 'Agent Thắng'}
        if self.move_count >= MAX_MOVES: return self._obs(), 0.0, True, False, {'is_success': False, 'reason': 'Hòa'}
        opp_a = self._get_opponent_move()
        self.last_opp_move = opp_a
        ox, oy = divmod(opp_a, BOARD_SIZE)
        self.board[ox, oy] = self.bot_player
        self._occupied[ox, oy] = True
        self.active_cells.add((ox, oy))
        self.move_count += 1
        if check_win(self.board, ox, oy, self.bot_player): return self._obs(), REWARD_LOSE, True, False, {'is_success': False, 'reason': 'Bot Thắng'}
        if self.move_count >= MAX_MOVES: return self._obs(), 0.0, True, False, {'is_success': False, 'reason': 'Hòa'}
        current_score = static_evaluate(self.board, self.agent_player, self.active_cells)
        progress_reward = current_score - self.last_score
        self.last_score = current_score
        block_bonus = REWARD_BLOCK * (1 + self.opponent_level * 0.1) if is_blocking else 0.0
        if self.opponent_level >= 7: shaped = 0.0
        elif self.opponent_level >= 5:
            fade = max(0.0, 1.0 - (self.opponent_level - 5) * 0.4)
            cap = max(0.05, 0.3 * fade)
            shaped = float(np.clip((progress_reward * REWARD_SCALE * fade) + (block_bonus * fade), -cap, cap))
        else:
            shaped = float(np.clip((progress_reward * REWARD_SCALE) + block_bonus, -1.0, 1.0))
        return self._obs(), shaped + STEP_PENALTY, False, False, {}

    def action_masks(self):
        if not self.active_cells:
            self._dilated_flat.fill(False)
            self._dilated_flat.reshape(BOARD_SIZE, BOARD_SIZE)[15:25, 15:25] = True
            return self._dilated_flat
        dilated2d = binary_dilation(self._occupied, structure=_DILATION_STRUCT)
        np.logical_and(dilated2d, ~self._occupied, out=dilated2d)
        self._dilated_flat[:] = dilated2d.ravel()
        return self._dilated_flat

def make_env(opponent_level=0):
    def _init():
        env = CaroEnv(opponent_level=opponent_level)
        return Monitor(env, info_keywords=('is_success',))
    return _init

print('✅ Environment OK')

In [ ]:
# =====================================================================
# CELL 4: Architecture (giữ nguyên CaroResNet)
# =====================================================================
class _ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(x + self.net(x))

class CaroResNet(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512, n_blocks=6):
        super().__init__(observation_space, features_dim)
        in_ch = observation_space.shape[0]
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.blocks = nn.Sequential(*[_ResBlock(64) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((8, 8)), nn.Flatten(),
            nn.Linear(64 * 8 * 8, features_dim), nn.ReLU(inplace=True)
        )
    def forward(self, obs): return self.head(self.blocks(self.stem(obs.float())))

print('✅ Architecture OK')

In [ ]:
# =====================================================================
# CELL 5: Callbacks — FIX AdaptiveLR + giữ nguyên các callback khác
# =====================================================================
CSV_HEADER = ['timestep','mean_reward','mean_ep_len','win_rate','policy_loss','value_loss',
              'entropy_loss','approx_kl','clip_fraction','fps','elapsed_min','opp_level',
              'learning_rate','ent_coef']

class TrainingLogger(BaseCallback):
    def __init__(self, csv_path, log_freq=1, verbose=1):
        super().__init__(verbose)
        self.csv_path, self.log_freq = csv_path, log_freq
        self._count, self._t0 = 0, time.time()
        # Append vào CSV cũ nếu đã có, không ghi đè
        if not os.path.exists(csv_path):
            with open(csv_path, 'w', newline='') as f: csv.writer(f).writerow(CSV_HEADER)
        else:
            print(f'  📋 Resume CSV: {csv_path}')

    def _on_rollout_end(self):
        self._count += 1
        if self._count % self.log_freq != 0: return
        buf = self.model.ep_info_buffer
        elapsed = max(time.time() - self._t0, 1)
        curr_lr = float(self.model.policy.optimizer.param_groups[0]['lr'])  # ← đọc đúng
        row = [
            self.num_timesteps,
            round(float(np.mean([e.get('r',0) for e in buf])) if buf else 0, 6),
            round(float(np.mean([e.get('l',0) for e in buf])) if buf else 0, 2),
            round(sum(1 for e in buf if e.get('is_success')) / len(buf) if buf else 0, 4),
            self.logger.name_to_value.get('train/policy_gradient_loss'),
            self.logger.name_to_value.get('train/value_loss'),
            self.logger.name_to_value.get('train/entropy_loss'),
            self.logger.name_to_value.get('train/approx_kl'),
            self.logger.name_to_value.get('train/clip_fraction'),
            int(self.num_timesteps / elapsed),
            round(elapsed / 60, 2),
            self.training_env.get_attr('opponent_level')[0],
            curr_lr,
            self.model.ent_coef,
        ]
        with open(self.csv_path, 'a', newline='') as f: csv.writer(f).writerow(row)
        if self.verbose >= 1:
            print(f'  [ResNet-L{row[11]}] step={self.num_timesteps:>10,} | win={row[3]*100:>5.1f}% | kl={row[7] or 0:.2e} | lr={curr_lr:.2e} | fps={row[9]:>5}')

    def _on_step(self): return True


class EntropyAnnealCallback(BaseCallback):
    def __init__(self, total_steps, ent_start=0.05, ent_end=0.005):
        super().__init__()
        self.total_steps, self.ent_start, self.ent_end = total_steps, ent_start, ent_end
    def _on_step(self):
        progress = min(self.num_timesteps / self.total_steps, 1.0)
        base_ent = self.ent_start + progress * (self.ent_end - self.ent_start)
        if float(self.model.ent_coef) <= base_ent + 0.005:
            self.model.ent_coef = base_ent
        return True


class AdaptiveLRCallback(BaseCallback):
    """
    FIX so với version cũ:
    - Đọc LR từ optimizer.param_groups[0]['lr'] thay vì model.learning_rate
    - kl_threshold giảm xuống 0.02 (nhạy hơn)
    - min_lr giảm xuống 5e-6 (có thể giảm sâu hơn)
    """
    def __init__(self, kl_threshold=0.02, lr_factor=0.5, min_lr=5e-6, max_lr=1e-4, increase_cooldown=10):
        super().__init__()
        self.kl_threshold = kl_threshold
        self.lr_factor = lr_factor
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.increase_cooldown = increase_cooldown
        self._cooldown_counter = 0

    def _on_step(self): return True

    def _on_rollout_end(self):
        try: kl = float(self.logger.name_to_value.get('train/approx_kl', 0))
        except: return True

        # ✅ FIX: đọc LR thực tế từ optimizer, không dùng model.learning_rate
        actual_lr = float(self.model.policy.optimizer.param_groups[0]['lr'])
        self._cooldown_counter += 1

        if kl > self.kl_threshold and actual_lr > self.min_lr:
            new_lr = max(actual_lr * self.lr_factor, self.min_lr)
            self._cooldown_counter = 0
            print(f'   [🔧 AdaptiveLR] KL ({kl:.3f}) CAO -> Giảm LR: {actual_lr:.2e} -> {new_lr:.2e}')
            self._set_lr(new_lr)
        elif kl < self.kl_threshold * 0.3 and actual_lr < self.max_lr and self._cooldown_counter >= self.increase_cooldown:
            new_lr = min(actual_lr * 1.05, self.max_lr)
            self._set_lr(new_lr)
        return True

    def _set_lr(self, new_lr):
        """Set LR vào optimizer VÀ patch lr_schedule để SB3 không override."""
        for pg in self.model.policy.optimizer.param_groups:
            pg['lr'] = new_lr
        # SB3 gọi lr_schedule(progress) mỗi rollout để lấy LR → patch thành constant
        self.model.lr_schedule = lambda _: new_lr
        self.model.learning_rate = new_lr


class CurriculumCallback(BaseCallback):
    def __init__(self, min_steps=300_000, patience=5, collapse_threshold=500_000, verbose=1):
        super().__init__(verbose)
        self.min_steps, self.patience, self.collapse_threshold = min_steps, patience, collapse_threshold
        self.current_level, self.max_level = 0, 10
        self._step_at_level_start, self._patience_counter = 0, 0

    def _on_rollout_end(self):
        if self.current_level >= self.max_level: return True
        buf = self.model.ep_info_buffer
        if len(buf) < 100: return True
        win_rate = sum(1 for e in buf if e.get('is_success')) / len(buf)
        mean_reward = float(np.mean([e.get('r', 0) for e in buf]))
        steps_at_level = self.num_timesteps - self._step_at_level_start
        effective_threshold = self.collapse_threshold * (1 + self.current_level * 0.3)
        if self.current_level >= 2 and steps_at_level > effective_threshold and win_rate < 0.10:
            self.current_level -= 1
            self._step_at_level_start, self._patience_counter = self.num_timesteps, 0
            self.training_env.env_method('set_opponent_level', self.current_level)
            self.model.ep_info_buffer.clear()
            print(f'\n🚨 ROLLBACK -> L{self.current_level} (Win={win_rate:.1%} sau {steps_at_level:,} steps)\n')
            return True
        if steps_at_level < self.min_steps: return True
        if win_rate >= 0.45 and mean_reward > -0.6:
            self._patience_counter += 1
            print(f'   [⏳ Patience: {self._patience_counter}/{self.patience}] Win={win_rate:.1%} | Rew={mean_reward:.2f}')
        else:
            self._patience_counter = 0
        if self._patience_counter >= self.patience:
            self.current_level += 1
            self._step_at_level_start, self._patience_counter = self.num_timesteps, 0
            self.training_env.env_method('set_opponent_level', self.current_level)
            self.model.ep_info_buffer.clear()
            if self.current_level >= 7:
                self.model.ent_coef = max(self.model.ent_coef, 0.02)
                print(f'   [🔥 Entropy Boost] ent_coef -> {self.model.ent_coef:.3f}')
            print(f'\n🎓 UPGRADE -> Level {self.current_level}\n')
        return True

    def _on_step(self): return True


class SaveCurriculumStateCallback(BaseCallback):
    def __init__(self, curriculum_cb, state_path):
        super().__init__()
        self.curriculum_cb, self.state_path = curriculum_cb, state_path
    def _on_rollout_end(self):
        np.save(self.state_path, {'level': self.curriculum_cb.current_level,
                                   'step_at_level_start': self.curriculum_cb._step_at_level_start,
                                   'steps_done': self.num_timesteps})
        return True
    def _on_step(self): return True

print('✅ Callbacks OK')

In [ ]:
# =====================================================================
# CELL 6: MAIN — Resume từ checkpoint 900K steps
# =====================================================================
TOTAL_STEPS = 20_000_000
NUM_ENVS    = 16

env = SubprocVecEnv([make_env(opponent_level=0) for _ in range(NUM_ENVS)])

policy_kwargs = dict(
    normalize_images=False,
    features_extractor_class=CaroResNet,
    features_extractor_kwargs=dict(features_dim=512, n_blocks=6),
    net_arch=dict(pi=[256, 128], vf=[256, 128])
)

curriculum_cb = CurriculumCallback(verbose=1)

print(f'♻️  RESUME từ {MODEL_PATH}...')
# ✅ FIX: learning_rate khởi đầu thấp hơn (1e-4 thay vì 2e-4)
model = MaskablePPO.load(MODEL_PATH, env=env, device='cuda')
model.ep_info_buffer = deque(maxlen=300)

# Load curriculum state
state = np.load(STATE_PATH, allow_pickle=True).item()
curriculum_cb.current_level       = state['level']
curriculum_cb._step_at_level_start = state['step_at_level_start']
steps_done = state['steps_done']
env.env_method('set_opponent_level', curriculum_cb.current_level)

# ✅ FIX: Set LR thấp hơn khi resume VÀ patch lr_schedule
resume_lr = 5e-5
for pg in model.policy.optimizer.param_groups: pg['lr'] = resume_lr
model.lr_schedule = lambda _: resume_lr  # patch để SB3 không override
model.learning_rate = resume_lr

remaining = max(TOTAL_STEPS - steps_done, 2_000_000)
print(f'  Level: {curriculum_cb.current_level} | Steps done: {steps_done:,} | Remaining: {remaining:,}')
print(f'  Resume LR: {resume_lr:.2e} (thấp hơn để ổn định KL trước)')
print(f'Policy params: {sum(p.numel() for p in model.policy.parameters()):,}')

callbacks = CallbackList([
    # ✅ FIX: kl_threshold=0.02, min_lr=5e-6, max_lr=1e-4
    AdaptiveLRCallback(kl_threshold=0.02, lr_factor=0.5, min_lr=5e-6, max_lr=1e-4),
    TrainingLogger(csv_path=CSV_PATH),
    CheckpointCallback(save_freq=max(100_000 // NUM_ENVS, 1), save_path=SAVE_DIR, name_prefix='caro_resnet'),
    EntropyAnnealCallback(TOTAL_STEPS),
    curriculum_cb,
    SaveCurriculumStateCallback(curriculum_cb, STATE_PATH),
])

try:
    model.learn(total_timesteps=remaining, callback=callbacks, reset_num_timesteps=False)
except KeyboardInterrupt:
    print('\n🛑 Dừng tay.')
finally:
    model.save(MODEL_PATH)
    print(f'💾 Saved to {MODEL_PATH}')